# EduPredict — Model Training Notebook
Trains Linear Regression, Decision Tree, Random Forest, Gradient Boosting and compares them.

In [ ]:
import pandas as pd, numpy as np, os, sys
BASE=os.path.abspath(os.path.join(os.getcwd(),'..'))
sys.path.insert(0,os.path.join(BASE,'src'))
from feature_engineering import feature_engineering
print('Running feature engineering...')
feature_engineering(os.path.join(BASE,'data','student_dataset.csv'), os.path.join(BASE,'data','student_dataset_featured.csv'))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error
import joblib

FEATURE_COLS=['Hours_Studied','Attendance','Previous_Scores','Test_Score','Project_Marks','Submission_Timeliness','Participation','Extra_C','Backlogs','engagement_feature','risk_feature','balance_feature','activeness_feature']

df=pd.read_csv(os.path.join(BASE,'data','student_dataset_featured.csv'))
X=df[FEATURE_COLS]; y=df['Exam_Score']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print('Train:',len(X_train),' Test:',len(X_test))

In [ ]:
models={'Linear Regression':LinearRegression(),'Decision Tree':DecisionTreeRegressor(random_state=42),'Random Forest':RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(n_estimators=100,random_state=42)}
results=[]
for name,mdl in models.items():
    mdl.fit(X_train,y_train); pred=mdl.predict(X_test)
    mae=round(mean_absolute_error(y_test,pred),3); rmse=round(mean_squared_error(y_test,pred)**0.5,3); r2=round(r2_score(y_test,pred),4)
    results.append({'Model':name,'R2':r2,'MAE':mae,'RMSE':rmse}); print(f'{name:25s} R2={r2} MAE={mae}')
results_df=pd.DataFrame(results).sort_values('R2',ascending=False)
print(results_df)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
colors=['#10b981','#3b82f6','#8b5cf6','#888']
axes[0].barh(results_df['Model'],results_df['R2'],color=colors); axes[0].set_title('R² Score'); axes[0].set_xlim(0,1)
axes[1].barh(results_df['Model'],results_df['MAE'],color=colors[::-1]); axes[1].set_title('MAE (lower better)')
plt.tight_layout(); plt.savefig(os.path.join(BASE,'outputs','graphs','model_comparison.png'),dpi=150); plt.show()

In [ ]:
best_name=results_df.iloc[0]['Model']
best_model=models[best_name]
model_path=os.path.join(BASE,'models','trained_model.pkl')
joblib.dump(best_model,model_path)
print(f'Best model: {best_name} saved to {model_path}')